# The UK's 2020 A-Level Grading Algorithm: When Group History Overrides the Individual

In this notebook I look at Ofqual's 2020 A-level grading algorithm as an eleventh mechanism behind an AI ethics failure: a system that, by design, let a school's past results override an individual student's own predicted grade, with how much override applied quietly depending on how many students sat that subject at that school.

In this notebook, I will:

- Summarize what the algorithm did, and why it provoked a national reversal within days
- Simulate two schools with identical student ability this year, differing only in how many students sat the subject
- Measure how much more one group loses than the other, purely from that difference in cohort size
- Try the fix the UK government actually used, and see what it gives up in exchange for fixing the unfairness

## 1. Background: What Was Reported

In 2020, UK exams were cancelled because of the COVID-19 pandemic, and England's exams regulator, Ofqual, needed a way to award final A-level and GCSE grades without anyone sitting an exam. Teachers submitted a predicted grade for each student, called a Centre Assessment Grade, along with a rank order of their students within each grade band for each subject. Because teachers' predicted grades were, in aggregate, substantially higher than the school's results in previous years, Ofqual built a statistical model to adjust those predictions, using each school's historical grade distribution in that subject over the previous three years, to keep the overall national results roughly in line with past years.

Crucially, the model didn't apply this adjustment evenly. For subjects with very few students at a given school, the model leaned heavily on the teacher's own prediction, because a historical distribution built from a handful of past students isn't statistically reliable enough to trust. For subjects with many students at a given school, the model leaned heavily on the school's historical distribution instead, overriding a meaningful share of individual students' predicted grades to make the current year's results match the school's past pattern.

When results were released, about 39% of A-level grades in England came out lower than the teacher's prediction, and analysis found the downgrades fell much harder on high-achieving students at larger, historically lower-performing schools, often in more disadvantaged areas, while smaller class sizes, more common at some private and selective schools, meant many of their students kept the grade their teacher had predicted. Public outcry from students, teachers, and universities followed within days, and on August 17, 2020, the UK government announced that students in England would receive whichever grade was higher, the algorithm's or the teacher's original prediction, effectively abandoning the algorithm's downgrades. Scotland's exams body had already reversed a similar algorithm-driven downgrade a week earlier, after its own public backlash.

## 2. Why This Is a Different Mechanism

This isn't a proxy feature standing in for a protected attribute, and it isn't a narrow or unrepresentative training set. The statistical idea behind the model, trust an individual prediction less when it comes from a small sample and trust a larger, more stable historical pattern more as the sample grows, is a completely ordinary, often reasonable piece of statistics on its own.

The problem is what that idea does once it meets the real world: cohort size wasn't picked to track anything about privilege, but it happened to correlate with it anyway, since smaller class sizes for a given subject were more common at some schools than others. A statistically defensible rule about trusting small samples less ended up functioning as a rule about trusting some schools' current students less than others, for reasons that had nothing to do with those students' actual ability this year.

## 3. A Note on the Simulation Below

This notebook does not use Ofqual's real grading model or its real historical grade distributions, which involved a far more detailed statistical procedure than what's modeled here. I use a simplified version of the same core idea, blending a predicted grade with a historical average, with the blend weighted by cohort size, to make one mechanism concrete: how two schools with identical student ability this year can end up with very different final grades, purely because of how many students sat the subject at each one.

## 4. Simulating Two Schools With Identical Student Ability

I generate students at two schools who have the exact same underlying ability this year, drawn from the same distribution, representing a genuinely strong year group at both. The only difference between the schools is how many students sat this subject, ten at one, two hundred at the other, mirroring how some schools simply have much smaller subject cohorts than others for reasons that have nothing to do with student quality.

In [ ]:
import random

random.seed(13)

N_STUDENTS_PER_SCHOOL = 2000
TRUE_ABILITY_MEAN = 68
TRUE_ABILITY_SD = 8
HISTORICAL_MEAN = 58
CAG_NOISE_SD = 3
TOP_GRADE_THRESHOLD = 66

SCHOOLS = [
    {"name": "Small-cohort school", "cohort_size": 10},
    {"name": "Large-cohort school", "cohort_size": 200},
]


def make_students(n, true_ability_mean, true_ability_sd, cag_noise_sd):
    """Returns a list of student dicts with a true ability and a teacher-predicted grade."""
    students = []
    for _ in range(n):
        true_ability = random.gauss(true_ability_mean, true_ability_sd)
        teacher_cag = true_ability + random.gauss(0, cag_noise_sd)
        students.append({"true_ability": true_ability, "teacher_cag": teacher_cag})
    return students

## 5. Modeling the Cohort-Size-Dependent Shrinkage

I model the algorithm's core idea directly: the smaller the cohort, the more weight goes on the teacher's prediction; the larger the cohort, the more weight goes on the school's historical average. Both schools here have the exact same historical average too, so any difference in outcome from here on is coming only from cohort size.

In [ ]:
def shrinkage_weight(cohort_size):
    """Returns the weight placed on the school's historical grade distribution for this cohort size."""
    return cohort_size / (cohort_size + 100)


for school in SCHOOLS:
    school["weight_on_history"] = shrinkage_weight(school["cohort_size"])
    print(school["name"], "- weight on historical average:", round(school["weight_on_history"], 3))

## 6. Measuring the Downgrade Each School Actually Receives

I compute each student's final grade as a blend of their teacher's prediction and the school's historical average, using the weight from the step above, and compare the average downgrade, and the share of students who would have earned a top grade on their teacher's prediction but lost it, between the two schools.

In [ ]:
for school in SCHOOLS:
    students = make_students(N_STUDENTS_PER_SCHOOL, TRUE_ABILITY_MEAN, TRUE_ABILITY_SD, CAG_NOISE_SD)
    w = school["weight_on_history"]
    for student in students:
        student["final_score"] = (1 - w) * student["teacher_cag"] + w * HISTORICAL_MEAN
        student["downgrade"] = student["teacher_cag"] - student["final_score"]

    avg_downgrade = sum(s["downgrade"] for s in students) / len(students)
    would_have_top_grade = [s for s in students if s["teacher_cag"] >= TOP_GRADE_THRESHOLD]
    lost_top_grade = [s for s in would_have_top_grade if s["final_score"] < TOP_GRADE_THRESHOLD]

    print(school["name"])
    print("  average downgrade from the teacher's predicted grade:", round(avg_downgrade, 2))
    print(
        "  share of would-have-top-grade students who lost it:",
        round(len(lost_top_grade) / len(would_have_top_grade), 3),
    )
    school["students"] = students

## 7. Trying the Fix the UK Government Actually Used

Facing exactly this disparity, the government's actual fix was simple: give every student whichever grade was higher, the algorithm's adjusted grade or the teacher's original prediction. I apply that same rule and recheck the average downgrade at both schools.

In [ ]:
for school in SCHOOLS:
    students = school["students"]
    for student in students:
        student["fixed_final"] = max(student["final_score"], student["teacher_cag"])
        student["fixed_downgrade"] = student["teacher_cag"] - student["fixed_final"]
    avg_fixed_downgrade = sum(s["fixed_downgrade"] for s in students) / len(students)
    print(school["name"], "- average downgrade after the fix:", round(avg_fixed_downgrade, 3))

## 8. What the Fix Gives Up in Exchange

The disparity between the two schools disappears, both are left with essentially no downgrade at all, because taking whichever grade is higher means almost every student simply keeps their teacher's prediction. That's the correct call for fairness between students, but it isn't free: it also throws out the entire statistical check the algorithm was originally built to provide, that teachers' predictions, in aggregate, were running well above what the school's results had ever supported before. Fixing the unfairness between individuals and fixing grade inflation at the national level turned out to be the same lever, pointed in opposite directions, and the UK government chose the former.